# Import necessary packages

In [ ]:
import geopandas as gpd
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import matplotlib as mpl

In [ ]:
# Implement this code in order to make any figures always use Times New Roman. Otherwise when you do it in the figures the matplotlib default can override it
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 12

# Set figure titles to be the same
mpl.rcParams['axes.titlesize'] = 20
mpl.rcParams['axes.titleweight'] = 'bold'
mpl.rcParams['axes.titlepad'] = 20
mpl.rcParams['font.family'] = 'Times New Roman'

## Set up base paths

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
networks_folder = base_path / "Processed_data/networks"

# Define the networks_catchments_intersections directory path
networks_catchments_intersections = base_path / "Processed_data/networks/networks_catchments_intersections"
networks_EAELs = base_path / "Processed_data/direct_damages_summary_uids"

In [ ]:
hydrobasins_path = base_path / "Processed_data/HydroBASINS_Level12_Clipped_Jamaica.shp"
catchments_gdf = gpd.read_file(hydrobasins_path)
print("Hydrobasins CRS:", catchments_gdf.crs)

# Set up output path

In [ ]:
networks_catchments_EAELs = base_path / "Processed_data/baseline_network_catchment_EAELs"

map_networks_damages_catchments_figures_folder = base_path / "Results/Map_Figures/networks_damages_catchments"

## Read in files 

In [ ]:
pipelines_edges_catchments_intersection = networks_catchments_intersections / "pipelines_edges_catchments_intersection.gpkg"

# Read the file into a GeoDataFrame
pipelines_edges_catchments_intersection = gpd.read_file(pipelines_edges_catchments_intersection)


In [ ]:
pipelines_edges_EAELs = networks_EAELs / "pipelines_NWC_edges_EAD_EAEL.parquet"
# Instead of using gpd.read_parquet, load the file with pandas:
pipelines_edges_EAELs_df = pd.read_parquet(pipelines_edges_EAELs)

In [ ]:
jamaica_metric_grid_crs = "EPSG:3448"

In [ ]:
jamaica_boundary_path = base_path / "Inputs/Boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(f"Original Jamaica boundary CRS: {jamaica_boundary.crs}")

## Filter for fluvial baseline 

In [ ]:
pipelines_edges_EAELs_fluvial_filtered = pipelines_edges_EAELs_df.query('hazard == "fluvial" and rcp == "baseline"')


In [ ]:
pipelines_edges_EAELs_fluvial_filtered

## Merge networks_catchments and the edges_EAELs data 

In [ ]:
# Merge the two layers based on the adjusted 'edge_id'
merged_edges = pipelines_edges_catchments_intersection.merge(
    pipelines_edges_EAELs_fluvial_filtered,
    on="edge_id",
    how="left"
)

In [ ]:
print("Merged columns:", merged_edges.columns)
print("Merged GeoDataFrame type:", type(merged_edges))

In [ ]:
# Show all columns in the output
pd.set_option('display.max_columns', None)
display(merged_edges)

## Summarise network information and damages by Hybas ID 

In [ ]:
# Build aggregation dictionary and group by HYBAS_ID
edges_cols_to_agg = [
    'EAD_undefended_amin', 
    'EAD_undefended_mean', 
    'EAD_undefended_amax',
    'EAEL_undefended_amin', 
    'EAEL_undefended_mean', 
    'EAEL_undefended_amax'
]

edges_agg_dict = {
    'edge_id': 'count',
    'Length': 'sum'
}
for col in edges_cols_to_agg:
    edges_agg_dict[col] = ['sum']

edges_summary = merged_edges.groupby("HYBAS_ID").agg(edges_agg_dict).reset_index()

# Flatten MultiIndex columns
edges_summary.columns = [
    '_'.join(filter(None, col)) if isinstance(col, tuple) else col
    for col in edges_summary.columns
]

# Rename columns for clarity
edges_summary = edges_summary.rename(columns={
    'edge_id_count': 'count_edges',
    'Length_sum': 'total_length_m'
})

# Merge the entire summary with catchments to add the geometry.
# This will include all aggregated columns.
edges_summary_with_geom = edges_summary.merge(
    catchments_gdf[['HYBAS_ID', 'geometry']], on='HYBAS_ID', how='left'
)

# Display and export the full summary table
display(edges_summary)
edges_summary.to_csv("water_pipelines_edges_baseline_EAD_EAEL_summary_table.csv", index=False)

In [ ]:
# Top 5 HYBAS_IDs based on highest EAD_undefended_amax_sum
edges_top5_EAD = edges_summary_with_geom.sort_values(by='EAD_undefended_amax_sum', ascending=False).head(5)
display(edges_top5_EAD[['HYBAS_ID', 'EAD_undefended_amax_sum', 'geometry']])

In [ ]:
# Top 5 HYBAS_IDs based on highest EAEL_undefended_amax_sum
edges_top5_EAEL = edges_summary_with_geom.sort_values(by='EAEL_undefended_amax_sum', ascending=False).head(5)
display(edges_top5_EAEL[['HYBAS_ID', 'EAEL_undefended_amax_sum', 'geometry']])

In [ ]:
# Define common extents (same as in your working maps)
common_xlim = (593635.9271443005, 848114.2122774671)
common_ylim = (612896.835085367, 712816.0327676072)

In [ ]:
fig1, ax1 = plt.subplots(figsize=(18, 14), dpi=300)

# Plot background catchments and boundary for context
catchments_gdf.plot(ax=ax1, color="lightgrey", edgecolor="white")
jamaica_boundary.plot(ax=ax1, facecolor="none", edgecolor="black", linewidth=0.2, zorder=100)

# Set up colormap (using Reds) based on the EAD damage values
cmap = plt.get_cmap("Reds")
norm = mcolors.Normalize(
    vmin=edges_top5_EAD['EAD_undefended_amax_sum'].min(), 
    vmax=edges_top5_EAD['EAD_undefended_amax_sum'].max()
)

legend_handles = []
for idx, row in edges_top5_EAD.iterrows():
    # Since the value is already numeric, just use it directly
    value = row['EAD_undefended_amax_sum']
    color = cmap(norm(value))
    # Plot the catchment polygon
    gpd.GeoSeries(row['geometry']).plot(ax=ax1, color=color, edgecolor="black", linewidth=1.5, zorder=101)
    centroid = row['geometry'].centroid
    # Format the value as desired in the annotation (for example, with 2 decimals)
    ax1.annotate(f"{row['HYBAS_ID']}\n{value:,.2f}", 
                 xy=(centroid.x, centroid.y), ha="center", fontsize=10, fontweight="bold")
    patch = mpatches.Patch(color=color, label=f"{row['HYBAS_ID']} ({value:,.2f})")
    legend_handles.append(patch)

ax1.legend(handles=legend_handles, title="HYBAS ID and baseline EAD undefended max (J$)", 
           bbox_to_anchor=(0.5, -0.1), loc="upper center", ncol=3,
           frameon=False, fontsize=12, title_fontsize=14)

# Define scale bar and north arrow functions
def add_scale_bar(ax, length_km=20, location=(0.9, 0.79), linewidth=2, tick_height=0.01, label_offset=0.04, km_offset=0.01):
    x, y = location
    bar_half_length = 0.05
    ax.plot([x - bar_half_length, x + bar_half_length], [y, y],
            transform=ax.transAxes, color="black", linewidth=linewidth)
    for pos in [x - bar_half_length, x, x + bar_half_length]:
        ax.plot([pos, pos], [y - tick_height/2, y + tick_height/2],
                transform=ax.transAxes, color="black", linewidth=linewidth)
    ax.text(x - bar_half_length, y - tick_height - label_offset, "0",
            transform=ax.transAxes, ha="center", va="center", fontsize=10)
    ax.text(x, y - tick_height - label_offset, f"{int(length_km // 2)}",
            transform=ax.transAxes, ha="center", va="center", fontsize=10)
    ax.text(x + bar_half_length, y - tick_height - label_offset, f"{int(length_km)}",
            transform=ax.transAxes, ha="center", va="center", fontsize=10)
    ax.text(x + bar_half_length + km_offset, y, "km",
            transform=ax.transAxes, ha="left", va="center", fontsize=12)

def add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=12, label_offset=0.03):
    x, y = location
    ax.annotate("", xy=(x, y + size), xycoords="axes fraction",
                xytext=(x, y), textcoords="axes fraction",
                arrowprops=dict(facecolor="black", edgecolor="black", headwidth=10, headlength=15, width=5))
    ax.text(x, y + size + label_offset, "N",
            transform=ax.transAxes, fontsize=fontsize, fontweight="bold", ha="center", va="center", color="black")

# Add scale bar and north arrow
add_scale_bar(ax1, length_km=20, location=(0.9, 0.79))
add_north_arrow(ax1, location=(0.9, 0.85))

ax1.set_xlim(common_xlim)
ax1.set_ylim(common_ylim)
ax1.set_xlabel("Easting", fontsize=14, fontname="Times New Roman")
ax1.set_ylabel("Northing", fontsize=14, fontname="Times New Roman")
ax1.set_title("Water pipelines edges: top 5 catchments by highest baseline EAD undefended max value", 
             fontsize=20, fontweight="bold", fontname="Times New Roman", pad=20)

plt.tight_layout()
fig1.savefig(map_networks_damages_catchments_figures_folder / "water_pipelines_edges_top5_baseline_EAD_max.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig2, ax2 = plt.subplots(figsize=(18, 14), dpi=300)

# Plot background catchments and boundary for context
catchments_gdf.plot(ax=ax2, color="lightgrey", edgecolor="white")
jamaica_boundary.plot(ax=ax2, facecolor="none", edgecolor="black", linewidth=0.2, zorder=100)

# Set up colormap (using Reds) based on the EAEL damage values
cmap = plt.get_cmap("Reds")
norm = mcolors.Normalize(
    vmin=edges_top5_EAEL['EAEL_undefended_amax_sum'].min(), 
    vmax=edges_top5_EAEL['EAEL_undefended_amax_sum'].max()
)

legend2_handles = []
for idx, row in edges_top5_EAEL.iterrows():
    # Use the numeric value directly
    value = row['EAEL_undefended_amax_sum']
    color = cmap(norm(value))
    # Plot the catchment polygon
    gpd.GeoSeries(row['geometry']).plot(ax=ax2, color=color, edgecolor="black", linewidth=1.5, zorder=101)
    centroid = row['geometry'].centroid
    ax2.annotate(f"{row['HYBAS_ID']}\n{value:,.2f}", 
                 xy=(centroid.x, centroid.y), ha="center", fontsize=10, fontweight="bold")
    patch = mpatches.Patch(color=color, label=f"{row['HYBAS_ID']} ({value:,.2f})")
    legend2_handles.append(patch)

ax2.legend(handles=legend2_handles, title="HYBAS ID and EAEL undefended max (J$)", 
           bbox_to_anchor=(0.5, -0.1), loc="upper center", ncol=3,
           frameon=False, fontsize=12, title_fontsize=14)

# Define scale bar and north arrow functions
def add_scale_bar(ax, length_km=20, location=(0.9, 0.79), linewidth=2, tick_height=0.01, label_offset=0.04, km_offset=0.01):
    x, y = location
    bar_half_length = 0.05
    ax.plot([x - bar_half_length, x + bar_half_length], [y, y],
            transform=ax.transAxes, color="black", linewidth=linewidth)
    for pos in [x - bar_half_length, x, x + bar_half_length]:
        ax.plot([pos, pos], [y - tick_height/2, y + tick_height/2],
                transform=ax.transAxes, color="black", linewidth=linewidth)
    ax.text(x - bar_half_length, y - tick_height - label_offset, "0",
            transform=ax.transAxes, ha="center", va="center", fontsize=10)
    ax.text(x, y - tick_height - label_offset, f"{int(length_km // 2)}",
            transform=ax.transAxes, ha="center", va="center", fontsize=10)
    ax.text(x + bar_half_length, y - tick_height - label_offset, f"{int(length_km)}",
            transform=ax.transAxes, ha="center", va="center", fontsize=10)
    ax.text(x + bar_half_length + km_offset, y, "km",
            transform=ax.transAxes, ha="left", va="center", fontsize=12)

def add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=12, label_offset=0.03):
    x, y = location
    ax.annotate("", xy=(x, y + size), xycoords="axes fraction",
                xytext=(x, y), textcoords="axes fraction",
                arrowprops=dict(facecolor="black", edgecolor="black", headwidth=10, headlength=15, width=5))
    ax.text(x, y + size + label_offset, "N",
            transform=ax.transAxes, fontsize=fontsize, fontweight="bold", ha="center", va="center", color="black")

# Add scale bar and north arrow to ax2
add_scale_bar(ax2, length_km=20, location=(0.9, 0.79))
add_north_arrow(ax2, location=(0.9, 0.85))

ax2.set_xlim(common_xlim)
ax2.set_ylim(common_ylim)
ax2.set_xlabel("Easting", fontsize=14, fontname="Times New Roman")
ax2.set_ylabel("Northing", fontsize=14, fontname="Times New Roman")
ax2.set_title("Water pipelines edges: top 5 catchments by highest baseline EAEL undefended max value", 
             fontsize=20, fontweight="bold", fontname="Times New Roman", pad=20)

plt.tight_layout()
fig2.savefig(map_networks_damages_catchments_figures_folder / "water_pipelines_edges_top5_baseline_EAEL_max.png", dpi=300, bbox_inches="tight")
plt.show()